# Ward heat — average land surface temperature for each of Bengaluru's 198 wards

For every ward this notebook averages the 30 m Landsat pixels inside the ward outline:
- `lst` — land surface temperature (°C), Jan–Apr 2022–2026 (same window as the other notebooks), **water pixels excluded**
- `ndvi`, `green` (share of pixels with NDVI > 0.4), `water` (share of the ward that is water) — for description only

**Before running:** have `bbmp_wards_198.geojson` (from `data/`) ready. The upload cell will ask for it.

Output: `bengaluru_ward_heat.csv` in your Google Drive. Put it in `data/`.

# SETUP

In [ ]:
import ee
import geemap.core as geemap

In [ ]:
PROJECT_ID = 'still-algebra-425406-v5'

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project=PROJECT_ID)

## Upload the ward outlines

In [ ]:
import json
from google.colab import files

uploaded = files.upload()   # choose bbmp_wards_198.geojson
wards_geojson = json.loads(uploaded['bbmp_wards_198.geojson'])
wards = ee.FeatureCollection(wards_geojson)
print('Wards:', wards.size().getInfo())   # should be 198

## Settings

In [ ]:
YEARS = (2022, 2026)   # dry seasons used
MONTHS = (1, 4)        # Jan-Apr
MAX_CLOUD = 60         # scene-level cloud %
GREEN_NDVI = 0.4

## Landsat 8/9: heat and greenness

In [ ]:
def prep(img):
    qa = img.select('QA_PIXEL')
    # remove dilated cloud (bit 1), cloud (bit 3), cloud shadow (bit 4)
    clear = (qa.bitwiseAnd(1 << 1).eq(0)
             .And(qa.bitwiseAnd(1 << 3).eq(0))
             .And(qa.bitwiseAnd(1 << 4).eq(0)))
    red = img.select('SR_B4').multiply(0.0000275).add(-0.2)
    nir = img.select('SR_B5').multiply(0.0000275).add(-0.2)
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('ndvi')
    return lst.addBands(ndvi).updateMask(clear).updateMask(img.select('ST_B10').gt(0))

area = wards.geometry().bounds()
landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(area)
           .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[1] + 1}-01-01')
           .filter(ee.Filter.calendarRange(MONTHS[0], MONTHS[1], 'month'))
           .filter(ee.Filter.lt('CLOUD_COVER', MAX_CLOUD)))
print('Landsat scenes used:', landsat.size().getInfo())

comp = landsat.map(prep).median()

## Water mask
Lakes are cool for reasons unrelated to parks, so water pixels are left out of the ward temperature.

In [ ]:
dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
      .filterBounds(area).filterDate('2025-01-01', '2026-01-01')
      .select('label').mode())
water = dw.eq(0).rename('water')
land = water.Not()

lst_land = comp.select('lst').updateMask(land).rename('lst')
ndvi_land = comp.select('ndvi').updateMask(land).rename('ndvi')
green = comp.select('ndvi').gt(GREEN_NDVI).updateMask(land).rename('green')

layers = ee.Image.cat([lst_land, ndvi_land, green, water]).toFloat()

## Quick look

In [ ]:
Map = geemap.Map()
Map.centerObject(wards, 11)
Map.addLayer(lst_land.clip(area),
             {'min': 28, 'max': 45, 'palette': ['040274', '30c8e2', '86e26e', 'ffff2b', 'ff9c2b', 'ff0000']},
             'LST (°C), land only')
Map.addLayer(wards.style(color='000000', fillColor='00000000', width=1), {}, 'Wards')
Map

## Average per ward and export to Drive

In [ ]:
stats = layers.reduceRegions(collection=wards, reducer=ee.Reducer.mean(), scale=30, tileScale=4)

task = ee.batch.Export.table.toDrive(
    collection=stats,
    description='bengaluru_ward_heat',
    fileFormat='CSV',
    selectors=['ward_num', 'ward_name', 'lst', 'ndvi', 'green', 'water'])
task.start()
print('Export started. It will appear in your Google Drive as bengaluru_ward_heat.csv')

In [ ]:
# Re-run this cell to check progress (READY -> RUNNING -> COMPLETED)
print(task.status()['state'], task.status().get('error_message', ''))